In [ ]:
import os
import glob
import numpy as np
import pandas as pd


from google import genai
from google.genai import types
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.2 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader

In [ ]:
from google import genai

In [ ]:
from google.colab import userdata

In [ ]:
GEMINI_API_KEY=userdata.get('gemini9')

In [ ]:
client=genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 01_Employee_Leave_Policy.pdf to 01_Employee_Leave_Policy.pdf
Saving 02_Work_From_Home_Policy.pdf to 02_Work_From_Home_Policy.pdf
Saving 03_Attendance_and_Working_Hours_Policy.pdf to 03_Attendance_and_Working_Hours_Policy.pdf
Saving 04_Travel_and_Reimbursement_Policy.pdf to 04_Travel_and_Reimbursement_Policy.pdf
Saving 05_Employee_Code_of_Conduct.pdf to 05_Employee_Code_of_Conduct.pdf
Saving 06_Performance_Management_Policy.pdf to 06_Performance_Management_Policy.pdf
Saving 07_Employee_Benefits_Policy.pdf to 07_Employee_Benefits_Policy.pdf
Saving 08_IT_and_Information_Security_Policy.pdf to 08_IT_and_Information_Security_Policy.pdf


In [ ]:
import os

os.makedirs("hr_documents", exist_ok=True)

for filename in uploaded.keys():
    os.rename(filename, os.path.join("hr_documents", filename))

print("Documents uploaded")

Documents uploaded


In [ ]:
def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    return pages

In [ ]:
pages = extract_pdf_text(
    "hr_documents/01_Employee_Leave_Policy.pdf"
)

print(pages[0]["text"])

Sample HR Policy Document | For RAG and training demonstration
Page 1
 EMPLOYEE LEAVE POLICY
Sample Organization | Human Resources
1. Purpose
This policy defines leave entitlements, eligibility, approval requirements, and procedures for employees.
2. Scope
This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable
law provides otherwise.
3. Annual Leave
Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be
requested at least three working days in advance.
4. Sick Leave
Employees may take up to 12 working days of sick leave per calendar year. A medical certificate may be requested
for extended or repeated absence.
5. Casual Leave
Employees may take up to 6 working days of casual leave per calendar year for personal or urgent matters.
6. Maternity and Paternity Leave
Eligible employees receive leave according to company rules and applicable statutory requirements. HR will
confirm the e

In [ ]:
pdf_files = glob.glob("hr_documents/*.pdf")

for file in pdf_files:
    print(file)

hr_documents/04_Travel_and_Reimbursement_Policy.pdf
hr_documents/08_IT_and_Information_Security_Policy.pdf
hr_documents/03_Attendance_and_Working_Hours_Policy.pdf
hr_documents/01_Employee_Leave_Policy.pdf
hr_documents/02_Work_From_Home_Policy.pdf
hr_documents/07_Employee_Benefits_Policy.pdf
hr_documents/05_Employee_Code_of_Conduct.pdf
hr_documents/06_Performance_Management_Policy.pdf


In [ ]:
documents = []

for pdf_path in pdf_files:

    pages = extract_pdf_text(pdf_path)

    for page in pages:

        documents.append({
            "source": os.path.basename(pdf_path),
            "page": page["page"],
            "text": page["text"]
        })

print("Total pages:", len(documents))

Total pages: 8


In [ ]:
def create_chunks(text, chunk_size=700, overlap=100):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
sample_text = documents[0]["text"]

chunks = create_chunks(sample_text)

print("Number of chunks:", len(chunks))

print(chunks[0])

Number of chunks: 1
Sample HR Policy Document | For RAG and training demonstration Page 1 TRAVEL AND REIMBURSEMENT POLICY Sample Organization | Human Resources 1. Purpose This policy defines approved business travel and reimbursement procedures. 2. Travel Approval Business travel must be approved by the employee's manager before booking, except where emergency business requirements apply. 3. Transportation Reasonable economy-class transportation should normally be used for domestic business travel unless an approved exception applies. 4. Accommodation Employees should use company-approved accommodation or accommodation within the applicable company limit. 5. Meals Reasonable meal expenses incurred during approved business travel may be reimbursed within company limits. 6. Local Transportation Business-related taxi, public transport, or approved mileage expenses may be reimbursed when supported by appropriate records. 7. Receipts Original or electronic receipts should be retained and su

In [ ]:
chunk_database = []

for doc in documents:

    chunks = create_chunks(doc["text"])

    for chunk_id, chunk in enumerate(chunks):

        chunk_database.append({
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks:", len(chunk_database))

Total chunks: 8


In [ ]:
pd.DataFrame(chunk_database).head()

,source,page,chunk_id,text
0,04_Travel_and_Reimbursement_Policy.pdf,1,0,Sample HR Policy Document | For RAG and traini...
1,08_IT_and_Information_Security_Policy.pdf,1,0,Sample HR Policy Document | For RAG and traini...
2,03_Attendance_and_Working_Hours_Policy.pdf,1,0,Sample HR Policy Document | For RAG and traini...
3,01_Employee_Leave_Policy.pdf,1,0,Sample HR Policy Document | For RAG and traini...
4,02_Work_From_Home_Policy.pdf,1,0,Sample HR Policy Document | For RAG and traini...


In [ ]:
def get_embedding(text):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    return np.array(result.embeddings[0].values)

In [ ]:
embedding = get_embedding(
    "How many annual leave days are available?"
)

print("Embedding dimensions:", len(embedding))

Embedding dimensions: 3072


In [ ]:
for i, item in enumerate(chunk_database):

    item["embedding"] = get_embedding(item["text"])

    if (i + 1) % 10 == 0:
        print("Embedded:", i + 1)

In [ ]:
len(chunk_database[0]["embedding"])

3072

In [ ]:
embedding_matrix = np.vstack([
    item["embedding"]
    for item in chunk_database
])

print(embedding_matrix.shape)

(8, 3072)


In [ ]:
def retrieve_documents(query, top_k=3):

    query_embedding = get_embedding(query)

    scores = cosine_similarity(
        [query_embedding],
        embedding_matrix
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:

        item = chunk_database[index].copy()

        item["score"] = float(scores[index])

        results.append(item)

    return results

In [ ]:
query = "How many annual leave days can an employee take?"

results = retrieve_documents(query)

for result in results:

    print("=" * 70)

    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Score:", round(result["score"], 4))

    print(result["text"])

Source: 01_Employee_Leave_Policy.pdf
Page: 1
Score: 0.6423
Sample HR Policy Document | For RAG and training demonstration Page 1 EMPLOYEE LEAVE POLICY Sample Organization | Human Resources 1. Purpose This policy defines leave entitlements, eligibility, approval requirements, and procedures for employees. 2. Scope This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable law provides otherwise. 3. Annual Leave Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be requested at least three working days in advance. 4. Sick Leave Employees may take up to 12 working days of sick leave per calendar year. A medical certificate may be requested for extended or repeated absence. 5. Casual Leave Employees may take up to 6 working days of casual leave per calendar year for personal or urgent matters. 6. Maternity and Paternity Leave Eligible employees receive leave according to company rules an

In [ ]:
def build_context(results):

    context = ""

    for i, result in enumerate(results, start=1):

        context += f"""
SOURCE {i}
Document: {result['source']}
Page: {result['page']}

Content:
{result['text']}

--------------------------------
"""

    return context

In [ ]:
def hr_rag(question, top_k=4):

    results = retrieve_documents(
        question,
        top_k=top_k
    )

    context = build_context(results)

    prompt = f"""
You are an HR Policy Assistant.

Answer the employee's question using ONLY the
HR policy context provided below.

Rules:

1. Do not invent company policies.
2. Do not use outside knowledge.
3. If the answer is not present in the context,
   say that the information was not found.
4. Give a concise and professional answer.
5. Mention the source document and page.
6. If the policy contains an exception, mention it.
7. Do not make legal conclusions.

HR POLICY CONTEXT:

{context}

EMPLOYEE QUESTION:

{question}

Provide:

Answer:
Source:
Policy Section or Relevant Content:
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text, results

In [ ]:
answer, sources = hr_rag(
    "How many annual leave days can an employee take?"
)

print(answer)

**Answer:**
Eligible employees receive 24 working days of annual leave per calendar year. 

*Exception:* This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable law provides otherwise. Additionally, leave during a notice period may require additional approval depending on business requirements and applicable employment terms.

**Source:** 
01_Employee_Leave_Policy.pdf, Page 1

**Policy Section or Relevant Content:** 
Section 3 (Annual Leave) and Section 2 (Scope) / Section 9 (Leave During Notice Period). 
Relevant text: *"Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be requested at least three working days in advance."*


In [ ]:
#testing